## Pro-Football Reference Coaches Scrape

### Helper Functions

In [2]:
import os
from bs4 import BeautifulSoup
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time
import re
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import requests

In [16]:
DIR = 'input/coaches/html'
DIR2 = 'input/teams'

In [4]:
def name(url):
    url = url.replace('-','_')
    return url.split('/')[-1]

In [5]:
def save(link,directory,sleep=10,name=name):
    save_path = os.path.join(directory, name(link))
    if not(os.path.exists(save_path)):
        time.sleep(sleep)
        response = requests.get(link);
        text = response.text
        with open(save_path, "w+") as f:
            f.write(text)
    else :
        with open(save_path, 'r') as f:
            text = f.read()
    return text

In [6]:
def save_tag(link,directory,tag,sleep=10,name=name):
    save_path = os.path.join(directory, name(link))
    if not(os.path.exists(save_path)):
        response = requests.get(link);
        text = response.text
        bs = BeautifulSoup(text, 'html.parser')
        text = bs.find(id = tag)
        time.sleep(sleep)
        with open(save_path, "w+") as f:
            f.write(str(text))
    else :
        with open(save_path, 'r') as f:
            text = f.read()
    return text

In [7]:
def toBS(text):
    bs = BeautifulSoup(text, 'html.parser')
    return bs

In [8]:
def save_tag(link,directory=DIR,tag="content",sleep=10,name=name):
    save_path = os.path.join(directory, name(link))
    if not(os.path.exists(save_path)):
        response = requests.get(link);
        text = response.text
        bs = BeautifulSoup(text, 'html.parser')
        text = bs.find(id = tag)
        time.sleep(sleep)
        with open(save_path, "w+") as f:
            f.write(str(text))
    else :
        with open(save_path, 'r') as f:
            text = f.read()
    return text

### Coach Specific Functions

In [9]:
def extractCoach(coachURL):
    return coachURL.split('/')[-1].split('.')[0]

In [10]:
def getCoach(code):
    return f"https://www.pro-football-reference.com/coaches/{code}.htm"

In [11]:
def nameTeam(url):
    splits = url.split('/')
    return splits[-2] + splits[-1]

In [12]:
def name2(url):
    splits = url.split('/')
    return splits[-2] + splits[-1] + '.html'

In [13]:
def hrefs(arr):
    base = 'https://www.pro-football-reference.com'
    return [base+a['href'] for a in arr]

In [18]:
# url = 'https://www.pro-football-reference.com/coaches/'
# name2(url)
# tag = "content"
# bs = toBS(save_tag(url,DIR,tag=tag,name=name2))
# a = hrefs(bs.find_all('a'))

In [ ]:
def fixCol(df,col,typ):
    df[col] = df[col].fillna(0).astype(typ)
    return df

In [ ]:
def resultsFooter(url):
    pass

In [19]:
def resultsPD(url,footerBool=False):
    bs = toBS(save_tag(url))
    df = pd.read_html(str(bs.find(id="all_coaching_results")))[0]
    # df.droplevel(0)
    df.columns = df.columns.get_level_values(1)
    footerRows = df['Year'].apply(lambda x: isinstance(x, str) and not x.isdigit())
    footer = df[footerRows]
    df = df[~footerRows]
    for col in ['Year','G plyf','W plyf','L plyf','Rank']:
        fixCol(df,col,int)
    if 'Num' not in df.columns :
        df.insert(len(df.columns)-1,'Num',0)
        df.insert(len(df.columns)-1,'Won',0)
    index = list(df.columns).index('W-L%',list(df.columns).index('W-L%')+1)
    df.columns.values[index] = 'W-L% plyf'
    for col in ['SRS', 'OSRS', 'DSRS','W-L% plyf','Num','Won'] :
        df[col] = df[col].fillna(0)
    # df['W-L% plyf'] = df['W-L% plyf'].fillna(0)
    df['Notes'] = df['Notes'].fillna("N/A")
    return df if not footerBool else footer

In [20]:
url = 'https://www.pro-football-reference.com/coaches/ShulDo0.htm'
df = pd.read_html(url)[0]
df.head()

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
                Year                Age                 Tm                 Lg   
0               1963                 33                BAL                NFL   
1               1964                 34                BAL                NFL   
2               1965                 35                BAL                NFL   
3               1966                 36                BAL                NFL   
4               1967                 37                BAL                NFL   

  Unnamed: 4_level_0 Unnamed: 5_level_0 Unnamed: 6_level_0 Unnamed: 7_level_0  \
                   G                  W                  L                  T   
0                 14                  8                  6                  0   
1                 14                 12                  2                  0   
2                 14                 10                  3                  1   
3                 14                  9                  5                  0   
4                 14                 11                  1                  2   

  Unnamed: 8_level_0 Simple Rating System           Playoffs                \
                W-L%                  SRS OSRS DSRS   G plyf W plyf L plyf   
0              0.571                  2.0  0.8  1.2      NaN    NaN    NaN   
1              0.857                 15.5  9.3  6.2      1.0    0.0    1.0   
2              0.769                 12.9  7.0  5.9      1.0    0.0    1.0   
3              0.643                  7.6  2.2  5.5      NaN    NaN    NaN   
4              0.917                 13.2  6.4  6.8      NaN    NaN    NaN   

  Unnamed: 15_level_0 Unnamed: 16_level_0 Unnamed: 17_level_0  
                 W-L%                Rank               Notes  
0                 NaN                 3.0                 NaN  
1                 0.0                 1.0                 NaN  
2                 0.0                 1.0                 NaN  
3                 NaN                 2.0                 NaN  
4                 NaN                 1.0                 NaN

In [21]:
print(getCoach('ShulDo0'))
df = resultsPD(getCoach('HalaGe0'))
# df18 = df[df['Year'] == 2018]
df.head()

https://www.pro-football-reference.com/coaches/ShulDo0.htm


FileNotFoundError: [Errno 2] No such file or directory: 'people/HalaGe0.htm'

In [19]:
# TODO: need to think about removing interim stints
# could look at team page and see if they are the first coach listed -> scrape would take time though

In [20]:
def addFeature(tenure=False):
    pass

In [ ]:
def coachDF(coachLink):
    df = resultsPD(coachLink)
    df.insert(0,'label',df['Year'].apply(lambda x: extractCoach(coachLink)+str(x%100)))
    df.loc[0, 'Tenure'] = 1
    df.loc[0,'Tenure_W'] = df.loc[0,'W']
    df.loc[0,'Tenure_L'] = df.loc[0,'L']
    df.loc[0,'Tenure_PW'] = df.loc[0,'W plyf']
    df.loc[0,'Tenure_PL'] = df.loc[0,'L plyf']
    df.loc[0,'Total_W'] = df.loc[0,'W']
    df.loc[0,'Total_L'] = df.loc[0,'L']
    df.loc[0,'Total_PW'] = df.loc[0,'W plyf']
    df.loc[0,'Total_PL'] = df.loc[0,'L plyf']
    df.loc[0,'Exp'] = 1
    df.loc[0, 'Fired'] = 0
    for i in range(1, len(df)):
        if not df.loc[i, 'Tm'] == df.loc[i-1, 'Tm']:
            df.loc[i, 'Tenure'] = 1
            df.loc[i,'Tenure_W'] = df.loc[i,'W']
            df.loc[i,'Tenure_L'] = df.loc[i,'L']
            df.loc[i,'Tenure_PW'] = df.loc[i,'W plyf']
            df.loc[i,'Tenure_PL'] = df.loc[i,'L plyf']
            df.loc[i-1, 'Fired'] = 1
        else:
            df.loc[i, 'Tenure'] = df.loc[i-1, 'Tenure'] + 1
            df.loc[i, 'Tenure_W'] = df.loc[i-1, 'Tenure_W'] + df.loc[i,'W']
            df.loc[i, 'Tenure_L'] = df.loc[i-1, 'Tenure_L'] + df.loc[i,'L']
            df.loc[i, 'Tenure_PW'] = df.loc[i-1, 'Tenure_PW'] + df.loc[i,'W plyf']
            df.loc[i, 'Tenure_PL'] = df.loc[i-1, 'Tenure_PL'] + df.loc[i,'L plyf']
            df.loc[i-1, 'Fired'] = 0
        df.loc[i,'Total_W'] = df.loc[i-1, 'Total_W']+ df.loc[i,'W']
        df.loc[i,'Total_L'] = df.loc[i-1, 'Total_L']+ df.loc[i,'L']
        df.loc[i,'Total_PW'] = df.loc[i-1, 'Total_PW']+ df.loc[i,'W plyf']
        df.loc[i,'Total_PL'] = df.loc[i-1, 'Total_PL']+ df.loc[i,'L plyf']
        df.loc[i,'Exp'] = i+1
#     TODO - hardcode coaches who were not fired, but instead retired
# TODO: fix coaches who stopped coaching in 2023
    df.loc[len(df)-1,'Fired'] = 1
    return df

Manually cleaning data for those coaches who retired and weren't fired:

In [ ]:
retired = [
    'CowhBi06',
    'DungTo08',
    'MaddJo078',
    'NollCh091',
    'LevyMa097',
    'LandTo088',
]

In [22]:
df_list = [coachDF(coach) for coach in a]
df_list

[        label  Year Age   Tm   Lg   G   W   L  T   W-L%  ...  Tenure_W  \
 0   ShulDo063  1963  33  BAL  NFL  14   8   6  0  0.571  ...       8.0   
 1   ShulDo064  1964  34  BAL  NFL  14  12   2  0  0.857  ...      20.0   
 2   ShulDo065  1965  35  BAL  NFL  14  10   3  1  0.769  ...      30.0   
 3   ShulDo066  1966  36  BAL  NFL  14   9   5  0  0.643  ...      39.0   
 4   ShulDo067  1967  37  BAL  NFL  14  11   1  2  0.917  ...      50.0   
 5   ShulDo068  1968  38  BAL  NFL  14  13   1  0  0.929  ...      63.0   
 6   ShulDo069  1969  39  BAL  NFL  14   8   5  1  0.615  ...      71.0   
 7   ShulDo070  1970  40  MIA  NFL  14  10   4  0  0.714  ...      10.0   
 8   ShulDo071  1971  41  MIA  NFL  14  10   3  1  0.769  ...      20.0   
 9   ShulDo072  1972  42  MIA  NFL  14  14   0  0  1.000  ...      34.0   
 10  ShulDo073  1973  43  MIA  NFL  14  12   2  0  0.857  ...      46.0   
 11  ShulDo074  1974  44  MIA  NFL  14  11   3  0  0.786  ...      57.0   
 12  ShulDo075  1975  45 

In [23]:
# index = list(df_list[0].columns).index('W-L%',list(df_list[0].columns).index('W-L%')+1)
# df_list[0].columns.values[index] = 'W-L% plyf'
df_list[0].columns

Index(['label', 'Year', 'Age', 'Tm', 'Lg', 'G', 'W', 'L', 'T', 'W-L%', 'SRS',
       'OSRS', 'DSRS', 'G plyf', 'W plyf', 'L plyf', 'W-L% plyf', 'Rank',
       'Num', 'Won', 'Notes', 'Tenure', 'Tenure_W', 'Tenure_L', 'Tenure_PW',
       'Tenure_PL', 'Total_W', 'Total_L', 'Total_PW', 'Total_PL', 'Exp',
       'Fired'],
      dtype='object')

In [24]:
df_concat = pd.concat(df_list, ignore_index=True)

In [25]:
df_concat.to_csv('coaches_full.csv',index=False)

In [29]:
df_concat.columns

Index(['label', 'Year', 'Age', 'Tm', 'Lg', 'G', 'W', 'L', 'T', 'W-L%', 'SRS',
       'OSRS', 'DSRS', 'G plyf', 'W plyf', 'L plyf', 'W-L% plyf', 'Rank',
       'Num', 'Won', 'Notes', 'Tenure', 'Tenure_W', 'Tenure_L', 'Tenure_PW',
       'Tenure_PL', 'Total_W', 'Total_L', 'Total_PW', 'Total_PL', 'Exp',
       'Fired'],
      dtype='object')

In [26]:
links = toBS(save_tag(a[0])).find(id="all_coaching_results").find_all('a')
teams = [t for t in hrefs(links) if "teams" in t and "_games" not in t]
# save_tag(teams[0],DIR2,'meta',name=nameTeam)

In [ ]:
def getYear(link):
    return int(link.split('/')[-1].split('.')[0])

In [ ]:
# Finish later!
for i in tqdm(a):
    save_tag(i)
    links = toBS(save_tag(i)).find(id="all_coaching_results").find_all('a')
    teams = [t for t in hrefs(links) if "teams" in t and "_games" not in t]
    for t in teams:
        yr = getYear(t);
        if yr > 2000:
            save_tag(t,DIR2,'meta',name=nameTeam)

100%|████████████████████████████████████████| 524/524 [00:02<00:00, 219.43it/s]


In [ ]:
DIR = 'people'
DIR2 = 'teams'

In [ ]:
url = 'https://www.pro-football-reference.com/coaches/'
save_tag(url,DIR,)